# Claude Skills 소개

Claude의 Skills 기능으로 전문적인 문서를 만들고, data를 분석하고, Excel, PowerPoint, PDF 생성을 사용하는 업무 workflow를 자동화하는 방법을 배웁니다.

> **실제 동작 보기:** 배울 Skills가 Claude의 파일 생성 능력을 뒷받침합니다! **[Claude의 파일 생성](https://www.anthropic.com/news/create-files)**에서 Skills가 Claude.ai 안에서 직접 문서를 만들고 편집하게 하는 방법을 확인하세요.

## 목차

1. [설정 및 설치](#setup)
2. [Skills 이해](#understanding)
3. [사용 가능한 Skills 찾기](#discovering)
4. [빠른 시작: Excel](#excel-quickstart)
5. [빠른 시작: PowerPoint](#powerpoint-quickstart)
6. [빠른 시작: PDF](#pdf-quickstart)
7. [문제 해결](#troubleshooting)

## 1. 설정 및 설치 {#setup}

### 사전 요구 사항

시작 전에 다음을 갖추었는지 확인하세요.
- Python 3.8 이상
- [console.anthropic.com](https://console.anthropic.com/)에서 받은 Anthropic API key

### 환경 설정 (처음 한 번만)

**아직 환경을 설정하지 않았다면** 다음 단계를 따르세요.

#### 1단계: Virtual Environment 생성

```bash
# skills directory로 이동
cd /path/to/claude-cookbooks/skills

# virtual environment 생성
python -m venv venv

# 활성화
source venv/bin/activate  # macOS/Linux에서
# 또는
venv\Scripts\activate     # Windows에서
```

#### 2단계: Dependency 설치

```bash
# venv 활성화 상태에서 requirements 설치
pip install -r requirements.txt
```

#### 3단계: VSCode/Jupyter에서 Kernel 선택

**VSCode:**
1. 이 notebook 열기
2. 오른쪽 위 kernel picker 클릭(예: "Python 3.11.x")
3. "Python Environments..." 선택
4. `./venv/bin/python` interpreter 선택

**Jupyter:**
1. Kernel menu → Change Kernel
2. venv에 맞는 kernel 선택

#### 4단계: API Key 설정

```bash
# 예제 파일 복사
cp .env.example .env

# .env를 편집하고 API key 추가:
# ANTHROPIC_API_KEY=sk-ant-api03-...
```

### 빠른 설치 확인

아래 셀로 환경이 올바르게 설정됐는지 확인하세요.

**위에 ❌ 또는 ⚠️ 경고가 있다면**, 계속하기 전에 설정 단계를 완료하세요.

**anthropic SDK 버전이 너무 오래됐다면(0.71.0 이상 필요):**
```bash
pip install anthropic>=0.71.0
```
그런 다음 새 버전을 사용하도록 **Jupyter kernel을 재시작**하세요.

---

### API 설정

이제 API key를 불러오고 client를 설정합니다.

### API 설정

**⚠️ 중요**: skills directory에 `.env` 파일을 만드세요.

```bash
# 예제 파일 복사
cp ../.env.example ../.env
```

그런 다음 `../.env`를 편집하여 Anthropic API key를 추가하세요.

In [ ]:
import os
import sys
from pathlib import Path

# import를 위해 path에 상위 directory 추가
sys.path.insert(0, str(Path.cwd().parent))


from anthropic import Anthropic
from dotenv import load_dotenv

# 파일 utility import
from file_utils import (
    download_all_files,
    extract_file_ids,
    get_file_info,
    print_download_summary,
)

# 상위 directory에서 환경 변수 로딩
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Copy ../.env.example to ../.env and add your API key."
    )

# client 초기화
# 참고: Skills 사용 시 요청별 beta header 추가
client = Anthropic(api_key=API_KEY)

# 없으면 outputs directory 생성
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print("\n📝 Note: Beta headers will be added per-request when using Skills")

### 연결 테스트

API 연결이 작동하는지 확인합니다.

In [ ]:
# API 연결 확인용 간단한 테스트
test_response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say 'Connection successful!' if you can read this.",
        }
    ],
)

print("API Test Response:")
print(test_response.content[0].text)
print(
    f"\n✓ Token usage: {test_response.usage.input_tokens} in, {test_response.usage.output_tokens} out"
)

## 2. Skills 이해 {#understanding}

### Skills란 무엇인가요?

**Skills**는 특정 작업에 특화된 능력을 Claude에 제공하는 지시, 실행 코드, resource로 구성된 package입니다. Claude가 동적으로 발견하고 불러올 수 있는 "전문성 package"라고 생각하세요.

📖 engineering 블로그 [Skills로 agent가 실제 세계에 대응할 수 있게 하기](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)를 읽어 보세요.

### Skills가 중요한 이유

MCP(Model Context Protocol)와 도구를 배운 뒤 Skills가 왜 중요한지 궁금할 수 있습니다.

- **Skills는 개별 도구보다 상위 수준**입니다. 지시, 코드, resource를 결합합니다.
- **Skills는 조합 가능**합니다. 여러 skill이 원활히 함께 작동합니다.
- **Skills는 효율적**입니다. progressive disclosure로 사용하는 것에만 비용을 냅니다.
- **Skills에는 검증된 코드가 포함**됩니다. 안정적으로 작동하는 helper script가 시간을 절약하고 오류를 줄입니다.

### 핵심 이점

- **전문가 수준 성능**: 학습 곡선 없이 전문적인 결과 제공
- **검증된 Helper Script**: Claude가 즉시 사용할 수 있는 테스트되고 작동하는 코드 포함
- **조직 지식**: 회사 workflow와 모범 사례를 package로 구성
- **비용 효율**: Progressive disclosure로 token 사용 최소화
- **신뢰성**: 미리 테스트한 script로 오류 감소와 일관된 결과
- **시간 절약**: Claude가 처음부터 코드를 생성하는 대신 기존 해결책 사용
- **조합 가능**: 복잡한 workflow를 위해 여러 skill이 함께 작동

### Progressive Disclosure Architecture

Skills는 3-tier loading model을 사용합니다.

![Progressive Disclosure - Skills 로딩 방식](../assets/prog-disc-1.png)

1. **Metadata**(name: 64자, description: 1024자): Claude는 skill 이름과 설명을 봅니다.
2. **전체 지시**(<5k token): 관련 skill이 있을 때 불러옵니다.
3. **연결된 파일**: 추가 resource는 필요할 때만 불러옵니다.

![Progressive Disclosure 단계](../assets/prog-disc-2.png)

필요에 따라 깊은 전문성을 제공하면서 효율적인 작동을 유지합니다. 처음에 Claude는 SKILL.md의 YAML frontmatter metadata만 봅니다. skill이 관련될 때만 helper script와 resource를 포함한 전체 내용을 불러옵니다.

### Skill 유형

| 유형 | 설명 | 예제 |
|------|-------------|----------|
| **Anthropic-Managed** | Anthropic이 유지 관리하는 사전 제작 skill | `xlsx`, `pptx`, `pdf`, `docx` |
| **Custom** | 특정 workflow를 위해 사용자가 정의한 skill | Brand 지침, 재무 model |

### Skills 개념 개요

![Skills 개념도](../assets/skills-conceptual-diagram.png)

이 diagram이 보여 주는 것:
- **Skill Directory 구조**: SKILL.md와 지원 파일로 Skills를 구성하는 방법
- **YAML Frontmatter**: Claude가 처음 보는 metadata
- **점진적 로딩**: Skills를 필요에 따라 발견하고 불러오는 방법
- **조합 가능성**: 단일 요청에서 여러 Skills가 함께 작동

### Skills와 Code Execution의 작동 방식

Skills는 **code execution** 도구가 활성화되어 있어야 합니다. 일반적인 workflow는 다음과 같습니다.

```python
# Skills 지원을 위해 client.beta.messages.create() 사용
response = client.beta.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=4096,
    container={
        "skills": [
            {"type": "anthropic", "skill_id": "xlsx", "version": "latest"}
        ]
    },
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[{"role": "user", "content": "Create an Excel file..."}],
    # extra_headers 대신 betas parameter 사용
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"]
)
```

**진행 과정:**
1. Claude는 xlsx skill이 불러와진 상태로 요청을 받음
2. Claude는 code execution으로 파일 생성
3. 응답에 생성한 파일의 `file_id` 포함
4. **Files API**로 파일 다운로드

**중요: Beta API**
- `client.beta.messages.create()` 사용(`client.messages.create()`가 아님)
- `container` parameter는 beta API에서만 사용 가능
- `betas` parameter로 beta 기능 활성화:
  - `code-execution-2025-08-25` - code execution 활성화
  - `files-api-2025-04-14` - 파일 다운로드에 필요
  - `skills-2025-10-02` - Skills 기능 활성화

⚠️ **참고**: Skills를 사용할 때 요청에 code_execution 도구를 반드시 포함해야 합니다.

### Token 사용 최적화

Skills는 프롬프트에 지시를 제공하는 것보다 token 사용을 크게 줄입니다.

| 접근 방식 | Token 비용 | 성능 |
|----------|------------|-------------|
| 수동 지시 | 요청당 5,000~10,000 token | 품질이 달라짐 |
| Skills (metadata만) | 최소(이름/설명만) | 전문가 수준 |
| Skills (전체 로딩) | skill 사용 시 약 5,000 token | 전문가 수준 |

**큰 이점:** 프롬프트를 부풀리지 않고 여러 skill을 넣을 수 있습니다. 실제 사용 전까지 각 skill은 metadata(이름 + 설명) 비용만 듭니다.

**예제**: 서식이 있는 Excel 파일 생성
- Skills 없음: 모든 Excel 기능을 먼저 설명하는 데 약 8,000 token
- Skills 있음: 처음에는 최소 metadata overhead, Excel skill 호출 시에만 약 5,000 token
- **핵심:** 98% 절감은 초기 context에 적용됩니다. skill을 사용하면 전체 지시가 로드됩니다.

**추가 이점:**
- Skills에는 작동이 확인된 helper script가 있어 신뢰성이 높아짐
- Claude가 처음부터 생성하는 대신 검증된 코드 패턴으로 시간 절약
- 더 일관되고 전문적인 결과 제공

### ⏱️ 예상 생성 시간

**⚠️ 중요**: Skills를 사용하는 문서 생성에는 code execution과 파일 생성이 필요하여 시간이 걸립니다. 인내심을 갖고 셀이 완료되게 하세요.

**관찰된 생성 시간:**
- **Excel 파일**: 약 2분(chart와 서식 포함)
- **PowerPoint presentation**: 약 1~2분(chart가 있는 간단한 2-slide presentation)
- **PDF 문서**: 약 40~60초(간단한 문서)

**예상되는 상황:**
- 실행 중 셀에 `[*]` 표시
- 1~2분 동안 "Executing..." 상태가 보일 수 있음
- **셀을 중단하지 마세요** - 완전히 완료되게 하세요.

**💡 권장 사항:**
1. **간단하게 시작**: 최소 예제로 설정 확인
2. **복잡성을 점진적으로 증가**: 기능을 조금씩 추가
3. **인내심 갖기**: 보통 40초~2분 소요
4. **참고**: 매우 복잡한 문서는 더 오래 걸릴 수 있음 - 예제의 범위를 집중적으로 유지

## 3. 사용 가능한 Skills 찾기 {#discovering}

### 모든 내장 Skills 나열

사용 가능한 Anthropic-managed skill을 찾아봅니다.

In [ ]:
# 사용 가능한 모든 Anthropic skill 나열
# 참고: Skills API에는 skills beta header 필요
client_with_skills_beta = Anthropic(
    api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"}
)

skills_response = client_with_skills_beta.beta.skills.list(source="anthropic")

print("Available Anthropic-Managed Skills:")
print("=" * 80)

for skill in skills_response.data:
    print(f"\n📦 Skill ID: {skill.id}")
    print(f"   Title: {skill.display_title}")
    print(f"   Latest Version: {skill.latest_version}")
    print(f"   Created: {skill.created_at}")

    # 버전 세부 사항 얻기
    try:
        version_info = client_with_skills_beta.beta.skills.versions.retrieve(
            skill_id=skill.id, version=skill.latest_version
        )
        print(f"   Name: {version_info.name}")
        print(f"   Description: {version_info.description}")
    except Exception as e:
        print(f"   (Unable to fetch version details: {e})")

print(f"\n\n✓ Found {len(skills_response.data)} Anthropic-managed skills")

### Skill Metadata 이해

각 skill에는 다음이 있습니다.
- **skill_id**: 고유 식별자(예: "xlsx", "pptx")
- **version**: 버전 번호 또는 "latest"
- **name**: 사람이 읽을 수 있는 이름
- **description**: skill이 하는 일
- **directory**: skill의 folder 구조

### Versioning 전략

- Anthropic skill은 `"latest"` 사용(권장)
- Anthropic이 skill을 자동 갱신
- production 안정성을 위해 특정 버전 고정
- Custom skill은 버전에 epoch timestamp 사용

### 예제: 월간 Budget Spreadsheet

간단한 한 줄과 상세한 요청, 두 예제로 시작합니다.

#### 간단한 예제 (1~2줄)
먼저 최소 프롬프트에서 Skills가 어떻게 작동하는지 봅니다.

```python
# 간단한 프롬프트 - Skills가 복잡성 처리
prompt = "Create a quarterly sales report Excel file with revenue data and a chart"
```

#### 상세 예제
더 제어하려면 구체적인 요구 사항을 제공할 수 있습니다.
- 수입과 지출 category
- 합계 formula
- 기본 서식

### 예제: 월간 Budget Spreadsheet

다음이 포함된 간단한 budget spreadsheet를 만듭니다.
- 수입과 지출 category
- 합계 formula
- 기본 서식

**⏱️ 참고**: Excel 생성은 보통 **1~2분** 걸립니다(chart와 서식 포함). 실행 중 셀에 `[*]`가 표시되므로 기다려 주세요!

In [ ]:
# Excel budget spreadsheet 생성
excel_response = client.beta.messages.create(  # 참고: Skills 지원을 위해 beta.messages 사용
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a monthly budget Excel spreadsheet with the following:

Income:
- Salary: $5,000
- Freelance: $1,200
- Investments: $300

Expenses:
- Rent: $1,500
- Utilities: $200
- Groceries: $600
- Transportation: $300
- Entertainment: $400
- Savings: $1,000

Include:
1. Formulas to calculate total income and total expenses
2. A formula for net savings (income - expenses)
3. Format currency values properly
4. Add a simple column chart showing income vs expenses
5. Use professional formatting with headers
""",
        }
    ],
    # beta 기능에 betas parameter 사용
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("Excel Response:")
print("=" * 80)
for content in excel_response.content:
    if content.type == "text":
        print(content.text)
    elif content.type == "tool_use":
        print(f"\n🔧 Tool: {content.name}")
        if hasattr(content, "input"):
            print(f"   Input preview: {str(content.input)[:200]}...")

print("\n\n📊 Token Usage:")
print(f"   Input: {excel_response.usage.input_tokens}")
print(f"   Output: {excel_response.usage.output_tokens}")

### Excel 파일 다운로드

이제 file_id를 추출하고 생성한 Excel 파일을 다운로드합니다.

In [ ]:
# 응답에서 file ID 추출
file_ids = extract_file_ids(excel_response)

if file_ids:
    print(f"✓ Found {len(file_ids)} file(s)\n")

    # 모든 파일 다운로드
    results = download_all_files(
        client, excel_response, output_dir=str(OUTPUT_DIR), prefix="budget_"
    )

    # 요약 출력
    print_download_summary(results)

    # 파일 세부 사항 표시
    for file_id in file_ids:
        info = get_file_info(client, file_id)
        if info:
            print("\n📄 File Details:")
            print(f"   Filename: {info['filename']}")
            print(f"   Size: {info['size'] / 1024:.1f} KB")
            print(f"   Created: {info['created_at']}")
else:
    print("❌ No files found in response")
    print("\nDebug: Response content types:")
    for i, content in enumerate(excel_response.content):
        print(f"  {i}. {content.type}")

**✨ 방금 무슨 일이 일어났나요?**

1. Claude가 `xlsx` skill로 전문적인 Excel 파일 생성
2. skill이 모든 Excel별 서식과 formula 처리
3. Claude의 code execution 환경에서 파일 생성
4. 응답에서 `file_id` 추출
5. Files API로 파일 다운로드
6. 파일이 `outputs/budget_*.xlsx`에 저장됨

Excel에서 파일을 열어 결과를 확인하세요!

## 5. 빠른 시작: PowerPoint {#powerpoint-quickstart}

이제 `pptx` skill로 PowerPoint presentation을 만듭니다.

### 예제: 매출 Presentation

#### 간단한 예제 (1줄)
```python
# 최소 프롬프트 - 세부 사항은 Skills에 맡김
prompt = "Create an executive summary presentation with 3 slides about Q3 results"
```

#### 상세 예제
**참고**: 생성 시간을 최소화하고 핵심 기능을 보여 주기 위해 의도적으로 간단하게 유지합니다(slide 2개, chart 1개).

### 예제: 간단한 매출 Presentation

**참고**: 생성 시간을 최소화하고 핵심 기능을 보여 주기 위해 의도적으로 간단하게 유지합니다(slide 2개, chart 1개).

In [ ]:
# PowerPoint presentation 생성
pptx_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pptx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple 2-slide PowerPoint presentation:

Slide 1: Title slide
- Title: "Q3 2025 Results"
- Subtitle: "Acme Corporation"

Slide 2: Revenue Overview
- Title: "Quarterly Revenue"
- Add a simple column chart showing:
  - Q1: $12M
  - Q2: $13M
  - Q3: $14M

Use clean, professional formatting.
""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PowerPoint Response:")
print("=" * 80)
for content in pptx_response.content:
    if content.type == "text":
        print(content.text)

print("\n\n📊 Token Usage:")
print(f"   Input: {pptx_response.usage.input_tokens}")
print(f"   Output: {pptx_response.usage.output_tokens}")

### PowerPoint 파일 다운로드

In [ ]:
# PowerPoint 파일 다운로드
file_ids = extract_file_ids(pptx_response)

if file_ids:
    results = download_all_files(
        client, pptx_response, output_dir=str(OUTPUT_DIR), prefix="q3_review_"
    )

    print_download_summary(results)

    print("\n✅ Open the presentation in PowerPoint or Google Slides to view!")
else:
    print("❌ No files found in response")

**⏱️ 참고**: 간단한 문서의 PDF 생성은 보통 **1~2분** 걸립니다. 실행 중 셀에 `[*]`가 표시되므로 기다려 주세요!

### 예제: PDF 문서

#### 간단한 예제 (1줄)
```python
# 빠른 PDF 생성
prompt = "Create a professional invoice PDF for $500 consulting services"
```

#### 상세 예제: 영수증
**참고**: 깔끔한 서식을 보장하기 위해 의도적으로 간단하게 유지합니다.

### 예제: 간단한 영수증

**참고**: 깔끔한 서식을 보장하기 위해 의도적으로 간단하게 유지합니다.

In [ ]:
# PDF 영수증 생성
pdf_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pdf", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple receipt PDF:

RECEIPT

Acme Corporation
Date: January 15, 2025
Receipt #: RCT-2025-001

Customer: Jane Smith

Items:
- Product A: $50.00
- Product B: $75.00
- Product C: $25.00

Subtotal: $150.00
Tax (8%): $12.00
Total: $162.00

Thank you for your business!

Use simple, clean formatting with clear sections.
""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PDF Response:")
print("=" * 80)
for content in pdf_response.content:
    if content.type == "text":
        print(content.text)

print("\n\n📊 Token Usage:")
print(f"   Input: {pdf_response.usage.input_tokens}")
print(f"   Output: {pdf_response.usage.output_tokens}")

### PDF 다운로드 및 검증

In [ ]:
# PDF 파일 다운로드
file_ids = extract_file_ids(pdf_response)

if file_ids:
    results = download_all_files(
        client, pdf_response, output_dir=str(OUTPUT_DIR), prefix="receipt_"
    )

    print_download_summary(results)

    # PDF 무결성 확인
    for result in results:
        if result["success"]:
            file_path = result["output_path"]
            file_size = result["size"]

            # 기본 PDF 검증
            with open(file_path, "rb") as f:
                header = f.read(5)
                if header == b"%PDF-":
                    print(f"\n✅ PDF file is valid: {file_path}")
                    print(f"   File size: {file_size / 1024:.1f} KB")
                else:
                    print(f"\n⚠️ File may not be a valid PDF: {file_path}")
else:
    print("❌ No files found in response")

## 7. 문제 해결 {#troubleshooting}

### 일반적인 문제와 해결책

### 문제 1: API Key를 찾지 못함

**오류:**
```
ValueError: ANTHROPIC_API_KEY not found
```

**해결책:**
1. 상위 directory에 `.env` 파일이 있는지 확인
2. `ANTHROPIC_API_KEY=sk-ant-api03-...` 설정 확인
3. .env를 생성/편집한 후 Jupyter kernel 재시작

### 문제 2: Container Parameter를 인식하지 못함

**오류:**
```
TypeError: Messages.create() got an unexpected keyword argument 'container'
```

**해결책:**
`client.messages.create()` 대신 `client.beta.messages.create()` 사용:
```python
# ✅ 올바름 - beta.messages 사용
response = client.beta.messages.create(
    model=MODEL,
    container={"skills": [...]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[...],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"]
)

# ❌ 잘못됨 - 일반 messages는 container 미지원
response = client.messages.create(
    model=MODEL,
    container={"skills": [...]},  # 오류!
    messages=[...]
)
```

### 문제 3: Skills Beta에 Code Execution Tool 필요

**오류:**
```
BadRequestError: Skills beta requires the code_execution tool to be included in the request.
```

**해결책:**
Skills를 사용할 때 code_execution 도구를 반드시 포함해야 합니다:
```python
# ✅ 올바름
response = client.beta.messages.create(
    model=MODEL,
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[...],
    betas=["...", "skills-2025-10-02"]
)

# ❌ 잘못됨 - code_execution 도구 누락
response = client.beta.messages.create(
    model=MODEL,
    messages=[...],
    betas=["...", "skills-2025-10-02"]
)
```

### 문제 4: 응답에서 파일을 찾지 못함

**오류:**
```
❌ No files found in response
```

**해결책:**
1. 요청에 code execution 도구가 포함됐는지 확인
2. skill 로딩 확인(응답 content 확인)
3. 작업에 실제로 파일 생성이 필요한지 확인
4. 응답 텍스트의 오류 message 확인

### 문제 5: 파일 다운로드 실패

**오류:**
```
Error retrieving file: File not found
```

**해결책:**
1. Anthropic server에서 파일의 보관 기간이 제한될 수 있음
2. 생성 직후 파일 다운로드
3. 응답에서 file_id가 올바르게 추출됐는지 확인
4. betas 목록에 Files API beta 포함 여부 확인

### Token 최적화 팁

1. Anthropic skill에 **"latest" 버전 사용** - 자동으로 최적화됨
2. **작업 일괄 처리** - 가능하면 한 대화에서 여러 파일 생성
3. **container 재사용** - skill 재로딩을 피하도록 이전 응답의 `container.id` 사용
4. **구체적으로 지시** - 명확한 지시로 iteration 감소

### API Rate Limiting

rate limit에 도달하면:
- retry에 exponential backoff 구현
- 여러 파일에 batch processing 사용
- 더 높은 limit을 위해 API tier 업그레이드 고려

## 다음 단계

🎉 **축하합니다!** Claude Skills의 기초를 배웠습니다.

### Skills의 실제 동작 보기

Skills가 Claude의 파일 생성 능력을 뒷받침하는 방법은 공식 발표에서 확인하세요.
- **[Claude의 파일 생성](https://www.anthropic.com/news/create-files)** - Skills로 Claude가 Excel, PowerPoint, PDF 파일을 직접 생성하고 편집하는 방법

### 계속 배우기:

- **[Notebook 2: 재무 Application](02_skills_financial_applications.ipynb)** - 재무 data를 사용하는 실제 업무 사용 사례
- **[Notebook 3: Custom Skills 개발](03_skills_custom_development.ipynb)** - 자신의 특화 skill 구축

### 지원 문서:

- 📚 **[Skills로 Claude에게 자신의 작업 방식 가르치기](https://support.claude.com/en/articles/12580051-teach-claude-your-way-of-working-using-skills)** - Skills 작업을 위한 사용자 가이드
- 🛠️ **[대화를 통해 Claude로 skill 만드는 방법](https://support.claude.com/en/articles/12599426-how-to-create-a-skill-with-claude-through-conversation)** - Interactive skill 생성 가이드

### Resource:

- [Claude API 문서](https://docs.anthropic.com/en/api/messages)
- [Skills 문서](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [Skills 모범 사례](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Files API 문서](https://docs.claude.com/en/api/files-content)
- [Claude Support](https://support.claude.com)

### 실험해 보세요:

1. 간단한 한 줄 프롬프트로 Skills 작동 확인
2. budget 예제를 수정해 category 추가
3. 자신의 data로 presentation 생성
4. 텍스트와 table을 결합한 PDF 보고서 생성
5. 단일 요청에서 여러 skill 함께 사용